# 04 – Data Visualizations
## Reel Insights: Global Movie Industry (TMDB)

This notebook creates 10 visualizations. Each chart is followed by an insight/recommendation block.


## 1. Imports, paths and data loading

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
DATA_DIR = os.path.join(PROJECT_DIR, "data")
CLEANED_DIR = os.path.join(DATA_DIR, "cleaned")
movies = pd.read_csv(os.path.join(CLEANED_DIR,"movies_cleaned.csv"))
genres = pd.read_csv(os.path.join(CLEANED_DIR,"genres_cleaned.csv"))
movie_genres = pd.read_csv(os.path.join(CLEANED_DIR,"movie_genres_cleaned.csv"))
cast = pd.read_csv(os.path.join(CLEANED_DIR,"cast_cleaned.csv"))
crew = pd.read_csv(os.path.join(CLEANED_DIR,"crew_cleaned.csv"))
movie_keywords = pd.read_csv(os.path.join(CLEANED_DIR,"movie_keywords_cleaned.csv"))
movies["release_date"] = pd.to_datetime(movies["release_date"], errors="coerce")
movies["release_year"] = movies.release_date.dt.year
sns.set_theme(style="whitegrid")

## Chart 1 – Movie releases per year

In [ ]:
year_counts = (movies.dropna(subset=["release_year"]).groupby("release_year")
               .movie_id.nunique().reset_index(name="movie_count"))
peak = year_counts.loc[year_counts.movie_count.idxmax()]
plt.figure(figsize=(14,6))
sns.lineplot(data=year_counts, x="release_year", y="movie_count", marker="o")
plt.title("Number of Movies Released Per Year"); plt.xlabel("Release Year")
plt.ylabel("Number of Movies"); plt.xticks(rotation=45)
plt.annotate(f"Peak: {int(peak.release_year)} ({int(peak.movie_count)} movies)",
             (peak.release_year, peak.movie_count), xytext=(10,15),
             textcoords="offset points", arrowprops={"arrowstyle":"->"})
plt.tight_layout(); plt.show()
print("Peak year:", int(peak.release_year), "| Movies:", int(peak.movie_count))

## Chart 2 – Number of movies by genre

In [ ]:
gd = movie_genres.merge(genres, on="genre_id")
gc = gd.groupby("genre_name").movie_id.nunique().reset_index(name="movie_count")
gc = gc.sort_values("movie_count", ascending=False)
plt.figure(figsize=(12,7)); sns.barplot(data=gc, x="movie_count", y="genre_name")
plt.title("Number of Movies by Genre"); plt.xlabel("Number of Movies"); plt.ylabel("Genre")
plt.tight_layout(); plt.show(); display(gc.head(5))
print("Movies represented in genres:", gd.movie_id.nunique())

## Chart 3 – Average popularity by primary genre

In [ ]:
primary = (movie_genres.sort_values(["movie_id","genre_id"])
           .drop_duplicates("movie_id", keep="first"))
pdg = primary.merge(genres, on="genre_id").merge(movies[["movie_id","popularity"]], on="movie_id")
ap = (pdg.groupby("genre_name").agg(average_popularity=("popularity","mean"),
     movie_count=("movie_id","nunique")).reset_index()
     .sort_values("average_popularity", ascending=False))
plt.figure(figsize=(12,7)); sns.barplot(data=ap, x="average_popularity", y="genre_name")
plt.title("Average Movie Popularity by Primary Genre")
plt.xlabel("Average Popularity"); plt.ylabel("Primary Genre")
plt.tight_layout(); plt.show(); display(ap.head(5))

## Chart 4 – Top 10 highest-grossing movies

In [ ]:
tr = movies[movies.revenue > 0].nlargest(10, "revenue")[["title","revenue"]].copy()
tr["revenue_billions"] = tr.revenue / 1e9
tr = tr.sort_values("revenue_billions")
plt.figure(figsize=(12,7)); sns.barplot(data=tr, x="revenue_billions", y="title")
plt.title("Top 10 Highest-Grossing Movies"); plt.xlabel("Revenue (USD Billions)")
plt.ylabel("Movie Title"); plt.tight_layout(); plt.show()
display(tr.sort_values("revenue_billions", ascending=False))

## Chart 5 – Top 10 movies by ROI

In [ ]:
r = movies[(movies.budget >= 1_000_000) & (movies.revenue > 0)].copy()
r["profit"] = r.revenue-r.budget
r["roi_percentage"] = r.profit/r.budget*100
toproi = r.nlargest(10, "roi_percentage")[["title","budget","revenue","roi_percentage"]]
plotroi = toproi.sort_values("roi_percentage")
plt.figure(figsize=(12,7)); sns.barplot(data=plotroi, x="roi_percentage", y="title")
plt.title("Top 10 Movies by Return on Investment (ROI)")
plt.xlabel("Return on Investment (%)"); plt.ylabel("Movie Title")
plt.tight_layout(); plt.show(); display(toproi)

## Chart 6 – Budget versus revenue

In [ ]:
fin = movies[(movies.budget > 0) & (movies.revenue > 0)].copy()
plt.figure(figsize=(12,7)); sns.scatterplot(data=fin, x="budget", y="revenue", alpha=.6)
plt.xscale("log"); plt.yscale("log")
lo = min(fin.budget.min(), fin.revenue.min()); hi = max(fin.budget.max(), fin.revenue.max())
plt.plot([lo,hi],[lo,hi],"--",label="Revenue = Budget")
plt.title("Relationship Between Movie Budget and Revenue")
plt.xlabel("Production Budget (USD - Log Scale)")
plt.ylabel("Revenue (USD - Log Scale)"); plt.legend()
plt.tight_layout(); plt.show()
print("Valid financial records:", len(fin))
print("Pearson correlation (raw values):", round(fin.budget.corr(fin.revenue),3))

## Chart 7 – Distribution of movie runtimes

In [ ]:
rt = movies.loc[movies.runtime > 0, "runtime"].dropna()
mean_rt, median_rt = rt.mean(), rt.median()
plt.figure(figsize=(12,6)); sns.histplot(rt, bins=30, kde=True)
plt.axvline(mean_rt, linestyle="--", label=f"Mean: {mean_rt:.1f} min")
plt.axvline(median_rt, linestyle="-.", label=f"Median: {median_rt:.1f} min")
plt.title("Distribution of Movie Runtimes"); plt.xlabel("Runtime (Minutes)")
plt.ylabel("Number of Movies"); plt.legend(); plt.tight_layout(); plt.show()
print("Count:",len(rt),"Mean:",round(mean_rt,2),"Median:",round(median_rt,2),
      "Min:",rt.min(),"Max:",rt.max())

## Chart 8 – Runtime outliers using IQR

In [ ]:
q1,q3 = rt.quantile(.25),rt.quantile(.75); iqr=q3-q1
lower,upper=q1-1.5*iqr,q3+1.5*iqr
out = rt[(rt<lower)|(rt>upper)]
plt.figure(figsize=(12,5)); sns.boxplot(x=rt)
plt.axvline(lower,linestyle="--",label=f"Lower: {lower:.1f}")
plt.axvline(upper,linestyle="--",label=f"Upper: {upper:.1f}")
plt.title("Movie Runtime Outlier Detection (IQR Method)")
plt.xlabel("Runtime (Minutes)"); plt.legend(); plt.tight_layout(); plt.show()
print("Q1:",q1,"Q3:",q3,"IQR:",iqr,"Bounds:",lower,upper)
print("Total:",len(out),"Short:",int((out<lower).sum()),"Long:",int((out>upper).sum()))

## Chart 9 – Top 10 most prolific actors

In [ ]:
ac = (cast.groupby(["person_id","actor_name"]).movie_id.nunique()
      .reset_index(name="movie_count").sort_values(
          ["movie_count","actor_name"],ascending=[False,True]).head(10))
ac_plot=ac.sort_values("movie_count")
plt.figure(figsize=(12,7)); sns.barplot(data=ac_plot,x="movie_count",y="actor_name")
plt.title("Top 10 Most Prolific Actors"); plt.xlabel("Number of Movies")
plt.ylabel("Actor"); plt.tight_layout(); plt.show(); display(ac)

## Chart 10 – Popularity and vote count

In [ ]:
pv = movies[(movies.popularity>0)&(movies.vote_count>0)].copy()
plt.figure(figsize=(12,7))
sns.regplot(data=pv,x="vote_count",y="popularity",
            scatter_kws={"alpha":.5},lowess=True)
plt.xscale("log"); plt.yscale("log")
plt.title("Relationship Between Movie Popularity and Vote Count")
plt.xlabel("Vote Count (Log Scale)"); plt.ylabel("Popularity (Log Scale)")
plt.tight_layout(); plt.show()
print("Movies analysed:",len(pv))
print("Pearson correlation (raw values):",round(pv.popularity.corr(pv.vote_count),3))

## 12. Overall visualization summary

The charts cover annual releases, genre distribution, genre popularity, revenue, ROI, budget/revenue relationship, runtime distribution and outliers, actor appearances, and popularity versus vote count.
